# 🏭 AI Project 2: Support Vector Classification (SVC)
## Kathmandu Valley Air Quality & Pollution Inversion Risk (2022 - 2025)
### Dataset Source: Kathmandu AQI Dataset by Subesh Yadav (Kaggle)

### Objective:
Classify hourly atmospheric pollution risk tiers (`Hazardous_Inversion`, `High_Stagnation`, `Moderate_Dispersion`, `Good_Ventilation`) in Kathmandu Valley from 2022 to 2025 using Support Vector Machines (SVC).

### SVC Mathematical Foundation:
Support Vector Classifier finds the optimal separating hyperplane that maximizes the margin $2/\|w\|$ while penalizing misclassifications:

$$\min_{w, b, \xi} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^n \xi_i$$
$$\text{subject to } y_i (w^T \phi(x_i) + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$

Where $\phi(x)$ maps features to infinite-dimensional Hilbert space via the Radial Basis Function (RBF) Kernel:
$$K(x, x') = \exp(-\gamma \|x - x'\|^2)$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

sns.set_theme(style="whitegrid")

## 1. Load and Inspect Kathmandu 2022-2025 Air Quality Dataset

In [ ]:
df = pd.read_csv('data/kathmandu_air_quality.csv')
print(f"Total Hourly Records: {len(df):,}")
df.head()

In [ ]:
print("Target Class Distribution (Risk Tiers):")
print(df['AQI_Risk_Level'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Season', y='Temperature_C', palette='Set2', ax=axes[0])
axes[0].set_title('Kathmandu Temperature (°C) by Season (2022-2025)', fontweight='bold')

sns.countplot(data=df, x='AQI_Risk_Level', palette='YlOrRd', ax=axes[1])
axes[1].set_title('Distribution of Kathmandu Pollution Risk Tiers', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Stratified Train-Test Split

In [ ]:
num_features = [
    'Temperature_C', 'Humidity_Pct', 'Apparent_Temp_C',
    'Wind_Speed_10m', 'Wind_Speed_100m', 'Wind_Shear',
    'Soil_Moisture_Surface', 'Soil_Moisture_Deep',
    'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos'
]
cat_features = ['Season']

X = df[num_features + cat_features]
y = df['AQI_Risk_Level'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

## 4. Train SVC with Different Kernels (Linear, RBF, Poly)

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    model = Pipeline([
        ('prep', preprocessor),
        ('svc', SVC(kernel=kernel, C=10.0, class_weight='balanced', random_state=42))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='macro')
    print(f"Kernel: {kernel.upper():<7} | Test Accuracy: {acc*100:.2f}% | Macro F1: {f1:.4f}")

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'svc__C': [1.0, 10.0, 50.0],
    'svc__gamma': ['scale', 0.05, 0.1]
}

grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))]),
    param_grid, cv=3, scoring='f1_macro', n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best Hyperparameters:", grid.best_params_)
best_svc = grid.best_estimator_

## 6. Evaluation & Confusion Matrix

In [ ]:
y_pred = best_svc.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

labels = ['Hazardous_Inversion', 'High_Stagnation', 'Moderate_Dispersion', 'Good_Ventilation']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Kathmandu Pollution Risk (2022-2025)', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 7. Interactive Prediction & Health Warning

In [ ]:
def check_inversion_risk(temp, hum, app_temp, w10, w100, soil_s, soil_d, hour, month, season):
    sample = pd.DataFrame([{
        'Temperature_C': temp,
        'Humidity_Pct': hum,
        'Apparent_Temp_C': app_temp,
        'Wind_Speed_10m': w10,
        'Wind_Speed_100m': w100,
        'Wind_Shear': w100 - w10,
        'Soil_Moisture_Surface': soil_s,
        'Soil_Moisture_Deep': soil_d,
        'Hour_Sin': np.sin(2 * np.pi * hour / 24.0),
        'Hour_Cos': np.cos(2 * np.pi * hour / 24.0),
        'Month_Sin': np.sin(2 * np.pi * month / 12.0),
        'Month_Cos': np.cos(2 * np.pi * month / 12.0),
        'Season': season
    }])
    pred = best_svc.predict(sample)[0]
    print(f"Atmospheric State: {temp}°C, {hum}% RH, {w10} km/h Wind ({season}, {hour}:00) -> Predicted Risk: {pred}")

check_inversion_risk(8.5, 92.0, 7.0, 1.8, 2.5, 0.38, 0.39, 8, 1, 'Winter')
check_inversion_risk(26.5, 65.0, 30.0, 14.0, 18.5, 0.42, 0.43, 14, 7, 'Monsoon')